In [2]:
import random
import numpy as np

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



data = load_diabetes()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


SEARCH_SPACE = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10]
}



def create_individual():

    return [
        random.choice(SEARCH_SPACE["n_estimators"]),
        random.choice(SEARCH_SPACE["max_depth"]),
        random.choice(SEARCH_SPACE["min_samples_split"])
    ]



def fitness(individual):

    n_estimators, max_depth, min_samples_split = individual

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)

    # Lower MAE = better fitness
    return mae




POPULATION_SIZE = 10
GENERATIONS = 10

population = [
    create_individual()
    for _ in range(POPULATION_SIZE)
]



for generation in range(GENERATIONS):

    # Evaluate population
    scores = []

    for individual in population:

        score = fitness(individual)

        scores.append((individual, score))

    # Sort according to MAE
    scores.sort(key=lambda x: x[1])

    # Best individual
    best_individual = scores[0][0]
    best_score = scores[0][1]

    print(
        f"Generation {generation + 1}: "
        f"Best MAE = {best_score:.4f}, "
        f"Parameters = {best_individual}"
    )



    parents = [
        individual
        for individual, score in scores[:POPULATION_SIZE // 2]
    ]



    new_population = parents.copy()

    while len(new_population) < POPULATION_SIZE:

        parent1 = random.choice(parents)
        parent2 = random.choice(parents)

        # Single-point crossover
        crossover_point = random.randint(1, 2)

        child = (
            parent1[:crossover_point]
            + parent2[crossover_point:]
        )

        # Mutation
        if random.random() < 0.2:

            parameter = random.randint(0, 2)

            if parameter == 0:
                child[0] = random.choice(
                    SEARCH_SPACE["n_estimators"]
                )

            elif parameter == 1:
                child[1] = random.choice(
                    SEARCH_SPACE["max_depth"]
                )

            elif parameter == 2:
                child[2] = random.choice(
                    SEARCH_SPACE["min_samples_split"]
                )

        new_population.append(child)

    population = new_population




final_scores = [
    (individual, fitness(individual))
    for individual in population
]

final_scores.sort(key=lambda x: x[1])

best_parameters = final_scores[0][0]
best_mae = final_scores[0][1]

print("\n========================================")
print("BEST HYPERPARAMETERS")
print("========================================")

print("n_estimators      :", best_parameters[0])
print("max_depth         :", best_parameters[1])
print("min_samples_split :", best_parameters[2])
print("Best MAE          :", round(best_mae, 4))



final_model = RandomForestRegressor(
    n_estimators=best_parameters[0],
    max_depth=best_parameters[1],
    min_samples_split=best_parameters[2],
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

predictions = final_model.predict(X_test)




mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(
    mean_squared_error(y_test, predictions)
)

r2 = r2_score(y_test, predictions)

print("\n========================================")
print("FINAL MODEL PERFORMANCE")
print("========================================")

print("MAE  :", round(mae, 4))
print("RMSE :", round(rmse, 4))
print("R²   :", round(r2, 4))

Generation 1: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 2: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 3: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 4: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 5: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 6: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 7: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 8: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 9: Best MAE = 42.7143, Parameters = [200, 3, 2]
Generation 10: Best MAE = 42.7143, Parameters = [200, 3, 2]

BEST HYPERPARAMETERS
n_estimators      : 200
max_depth         : 3
min_samples_split : 2
Best MAE          : 42.7143

FINAL MODEL PERFORMANCE
MAE  : 42.7143
RMSE : 52.6716
R²   : 0.4764
